# 03. Model Training

이 Notebook에서는 전처리가 완료된 mmBERT Token Classification Dataset을 이용하여 민감정보 탐지 모델을 학습하고 평가한다.

이전 단계에서 다음 Pipeline을 완료하였다.

- Structure-aware Block Builder
- Block-local Character Span 변환
- mmBERT Tokenizer 적용
- Character Span → BIO Label Alignment
- `max_length=1024`, `stride=256` 정책 확정
- Train / Validation / Test Tokenized Dataset 생성
- Tokenized Dataset 무결성 검증

이번 Notebook에서는 다음 작업을 진행한다.

1. mmBERT Baseline 모델 구성
2. 학습 Dataset 및 Dynamic Padding 구성
3. Baseline Fine-tuning
4. Validation 평가
5. Block-level Character Span 기반 성능 평가
6. Test 평가
7. Error Analysis
8. 성능 개선

## 29. mmBERT Baseline Token Classification

Primary Backbone으로 선정한 `jhu-clsp/mmBERT-base`에 Token Classification Head를 추가하여 25개의 BIO Label을 예측하는 Baseline 모델을 구성한다.

Baseline에서는 복잡한 성능 개선 기법을 적용하지 않는다.

먼저 기본 Cross Entropy Loss를 이용하여 모델이 현재 Dataset에서 어느 정도의 성능을 보이는지 확인한다.

Padding은 모든 Sequence를 1024 Token으로 고정하지 않고 Batch 내부의 가장 긴 Sequence에 맞추는 Dynamic Padding을 사용한다.

`-100` Label은 Loss 계산에서 제외한다.

### 29.1 Tokenized Dataset 및 Manifest 로드

학습 단계에서는 전처리 Notebook에서 저장한 mmBERT 전용 Tokenized Dataset을 파일에서 다시 로드한다.

이를 통해 이전 Notebook의 Python 변수에 의존하지 않고 학습 과정을 독립적으로 재현할 수 있도록 한다.

사용하는 Dataset은 다음과 같다.

- Train
- Validation
- Test

Test Dataset은 현재 학습이나 Hyperparameter 결정에 사용하지 않는다.

In [4]:
import accelerate

from transformers.utils import (
    is_accelerate_available,
    ACCELERATE_MIN_VERSION,
)


is_accelerate_available.cache_clear()


print(
    "Accelerate version :",
    accelerate.__version__,
)

print(
    "Required version   :",
    ACCELERATE_MIN_VERSION,
)

print(
    "Available          :",
    is_accelerate_available(),
)


assert is_accelerate_available()

Accelerate version : 1.14.0
Required version   : 1.1.0
Available          : True


In [5]:
from pathlib import Path
from collections import Counter

import json
import os

import numpy as np
import pandas as pd

In [6]:
TOKENIZED_DIR = Path(
    "../data/processed/tokenized/mmbert"
)


TRAIN_PATH = (
    TOKENIZED_DIR
    / "train.jsonl"
)

VALIDATION_PATH = (
    TOKENIZED_DIR
    / "validation.jsonl"
)

TEST_PATH = (
    TOKENIZED_DIR
    / "test.jsonl"
)

MANIFEST_PATH = (
    TOKENIZED_DIR
    / "manifest.json"
)

In [8]:
def load_jsonl(path):
    records = []

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:
        for line in f:
            if line.strip():
                records.append(
                    json.loads(line)
                )

    return records

In [9]:
train_records = load_jsonl(
    TRAIN_PATH
)

validation_records = load_jsonl(
    VALIDATION_PATH
)

test_records = load_jsonl(
    TEST_PATH
)


with MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as f:
    manifest = json.load(f)


print(
    f"Train      : "
    f"{len(train_records):,}"
)

print(
    f"Validation : "
    f"{len(validation_records):,}"
)

print(
    f"Test       : "
    f"{len(test_records):,}"
)

print(
    f"Labels     : "
    f"{manifest['num_labels']}"
)

print(
    f"Backbone   : "
    f"{manifest['backbone']}"
)

Train      : 44,626
Validation : 5,163
Test       : 5,385
Labels     : 25
Backbone   : jhu-clsp/mmBERT-base


### 29.2 BIO Label Mapping 복원

Token Classification Head의 출력 순서와 Dataset Label ID가 정확히 일치해야 한다.

전처리 단계에서 저장한 Manifest의 `label2id`, `id2label`을 그대로 모델 Config에 적용한다.

In [10]:
label2id = (
    manifest[
        "label2id"
    ]
)


id2label = {
    int(key): value
    for key, value
    in manifest[
        "id2label"
    ].items()
}


NUM_LABELS = (
    manifest[
        "num_labels"
    ]
)

PRIMARY_BACKBONE = (
    manifest[
        "backbone"
    ]
)

MAX_LENGTH = (
    manifest[
        "max_length"
    ]
)

IGNORE_INDEX = (
    manifest[
        "ignore_index"
    ]
)


print(
    f"Labels     : {NUM_LABELS}"
)

print(
    f"Max length : {MAX_LENGTH}"
)

print(
    f"Ignore idx : {IGNORE_INDEX}"
)

Labels     : 25
Max length : 1024
Ignore idx : -100


### 29.3 PyTorch 학습 Device 확인

현재 실행 환경에서 사용할 Accelerator를 확인한다.

우선순위는 다음과 같다.

1. CUDA
2. Apple Silicon MPS
3. CPU

MacBook의 Apple Silicon에서 MPS가 활성화되어 있다면 GPU 가속을 이용할 수 있다.

In [11]:
import torch
import transformers


if torch.cuda.is_available():
    device = torch.device(
        "cuda"
    )

elif (
    hasattr(
        torch.backends,
        "mps",
    )
    and torch.backends.mps.is_available()
):
    device = torch.device(
        "mps"
    )

else:
    device = torch.device(
        "cpu"
    )


print(
    f"PyTorch version      : "
    f"{torch.__version__}"
)

print(
    f"Transformers version : "
    f"{transformers.__version__}"
)

print(
    f"Device               : "
    f"{device}"
)

print(
    f"CUDA available       : "
    f"{torch.cuda.is_available()}"
)

print(
    f"MPS available        : "
    f"{torch.backends.mps.is_available()}"
)

PyTorch version      : 2.13.0
Transformers version : 5.15.0
Device               : mps
CUDA available       : False
MPS available        : True


### 29.4 Token Classification Dataset 구성

저장된 Tokenized Record에는 학습에 필요한 값 외에도 Character Offset과 Metadata가 포함되어 있다.

Baseline 모델의 `forward()`에 직접 필요한 값만 Dataset에서 반환한다.

사용하는 값은 다음 세 가지이다.

- `input_ids`
- `attention_mask`
- `labels`

`offset_mapping`, `source_sample_id` 등의 Metadata는 이후 Character Span 기반 평가 단계에서 다시 사용한다.

In [12]:
from torch.utils.data import Dataset


class TokenClassificationDataset(
    Dataset
):
    def __init__(
        self,
        records,
    ):
        self.records = records

    def __len__(
        self,
    ):
        return len(
            self.records
        )

    def __getitem__(
        self,
        index,
    ):
        record = (
            self.records[
                index
            ]
        )

        return {
            "input_ids":
                record[
                    "input_ids"
                ],

            "attention_mask":
                record[
                    "attention_mask"
                ],

            "labels":
                record[
                    "labels"
                ],
        }

In [13]:
train_dataset = (
    TokenClassificationDataset(
        train_records
    )
)

validation_dataset = (
    TokenClassificationDataset(
        validation_records
    )
)

test_dataset = (
    TokenClassificationDataset(
        test_records
    )
)


print(
    f"Train Dataset      : "
    f"{len(train_dataset):,}"
)

print(
    f"Validation Dataset : "
    f"{len(validation_dataset):,}"
)

print(
    f"Test Dataset       : "
    f"{len(test_dataset):,}"
)

Train Dataset      : 44,626
Validation Dataset : 5,163
Test Dataset       : 5,385


### 29.5 Dynamic Padding 구성

각 Sequence의 실제 길이는 서로 다르다.

따라서 모든 Sample을 `max_length=1024`로 Padding하지 않고, 각 Batch에서 가장 긴 Sequence에 맞춰 Dynamic Padding을 적용한다.

Token Classification에서는 Input뿐 아니라 Label도 동일한 길이로 Padding해야 한다.

Label Padding 값은 `-100`으로 설정하여 Loss 계산에서 제외한다.

In [14]:
from transformers import (
    AutoTokenizer,
    DataCollatorForTokenClassification,
)


tokenizer = (
    AutoTokenizer.from_pretrained(
        PRIMARY_BACKBONE,
        use_fast=True,
    )
)


data_collator = (
    DataCollatorForTokenClassification(
        tokenizer=tokenizer,
        padding="longest",
        label_pad_token_id=IGNORE_INDEX,
        return_tensors="pt",
    )
)

In [15]:
sample_batch = (
    data_collator(
        [
            train_dataset[0],
            train_dataset[1],
        ]
    )
)


print(
    "BATCH SHAPES"
)

print("=" * 50)


for key, value in (
    sample_batch.items()
):
    print(
        f"{key:<20} "
        f"{tuple(value.shape)}"
    )

BATCH SHAPES
input_ids            (2, 135)
attention_mask       (2, 135)
labels               (2, 135)


### 29.7 mmBERT Token Classification 모델 구성

사전학습된 mmBERT Encoder에 새로운 Token Classification Head를 추가한다.

출력 Label 수는 BIO Label 25개로 설정한다.

Encoder는 사전학습된 Weight를 사용하지만 Token Classification Head는 현재 프로젝트의 Label 수에 맞춰 새롭게 초기화된다.

따라서 모델 로드 시 Classification Head 일부 Weight가 새로 초기화되었다는 Warning이 나타나는 것은 정상이다.

In [16]:
from transformers import (
    AutoModelForTokenClassification,
)


model = (
    AutoModelForTokenClassification
    .from_pretrained(
        PRIMARY_BACKBONE,

        num_labels=NUM_LABELS,

        id2label=id2label,

        label2id=label2id,
    )
)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForTokenClassification LOAD REPORT from: jhu-clsp/mmBERT-base
Key               | Status     | 
------------------+------------+-
decoder.weight    | UNEXPECTED | 
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
total_parameters = sum(
    parameter.numel()
    for parameter
    in model.parameters()
)


trainable_parameters = sum(
    parameter.numel()
    for parameter
    in model.parameters()
    if parameter.requires_grad
)


print(
    f"Model class          : "
    f"{type(model).__name__}"
)

print(
    f"Total parameters     : "
    f"{total_parameters:,}"
)

print(
    f"Trainable parameters : "
    f"{trainable_parameters:,}"
)

print(
    f"Number of labels     : "
    f"{model.config.num_labels}"
)

Model class          : ModernBertForTokenClassification
Total parameters     : 307,549,465
Trainable parameters : 307,549,465
Number of labels     : 25


### 29.8 Forward Pass Smoke Test

전체 Fine-tuning을 시작하기 전에 하나의 작은 Batch를 모델에 입력하여 학습 Pipeline이 정상적으로 연결되는지 확인한다.

다음 항목을 검증한다.

- Device 이동
- Input Shape
- Logit Shape
- Loss 계산
- BIO Label 25개 출력

이 단계가 정상적으로 동작한 뒤 실제 TrainingArguments와 Trainer를 구성한다.

In [18]:
smoke_index = min(
    range(
        len(train_records)
    ),
    key=lambda index:
        len(
            train_records[
                index
            ]["input_ids"]
        ),
)


smoke_batch = (
    data_collator(
        [
            train_dataset[
                smoke_index
            ]
        ]
    )
)


print(
    "Smoke sequence length:",
    smoke_batch[
        "input_ids"
    ].shape[1],
)

Smoke sequence length: 13


In [19]:
model = model.to(
    device
)

model.eval()


device_batch = {
    key: value.to(
        device
    )
    for key, value
    in smoke_batch.items()
}


with torch.no_grad():
    outputs = model(
        **device_batch
    )


print(
    f"Device       : "
    f"{device}"
)

print(
    f"Loss         : "
    f"{outputs.loss.item():.6f}"
)

print(
    f"Logits shape : "
    f"{tuple(outputs.logits.shape)}"
)

Device       : mps
Loss         : 4.428921
Logits shape : (1, 13, 25)


### 29.9 Baseline 학습 정책

전체 학습에 앞서 Baseline Fine-tuning 설정을 정의한다.

Baseline의 목적은 복잡한 최적화 없이 현재 Dataset과 mmBERT가 어느 정도의 성능을 보이는지 기준 성능을 확보하는 것이다.

초기 학습 정책은 다음과 같다.

- Epoch: 3
- Learning Rate: 3e-5
- Train Batch Size: 2
- Gradient Accumulation: 8
- Effective Batch Size: 16
- Validation Batch Size: 2
- Optimizer: AdamW
- Weight Decay: 0.01
- Warmup Ratio: 0.1
- LR Scheduler: Linear
- Best Model 기준: Validation Loss
- Dynamic Padding 사용
- Mixed Precision 미사용
- Gradient Checkpointing 미사용

Apple Silicon MPS에서 최대 1024 Token의 Sequence를 처리해야 하므로 실제 Device Batch Size는 보수적으로 2부터 시작한다.

Gradient Accumulation을 이용해 모델 Weight Update 기준의 Effective Batch Size는 16으로 유지한다.

Baseline 단계에서는 안정성과 재현성을 우선하며, Batch Size, Mixed Precision, Gradient Checkpointing 등의 최적화는 이후 성능 및 학습 효율 개선 단계에서 비교한다.

In [20]:
from transformers import TrainingArguments


MODEL_OUTPUT_DIR = (
    "../models/mmbert_baseline"
)


training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,

    # -------------------------
    # Training
    # -------------------------
    num_train_epochs=3,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=8,

    learning_rate=3e-5,
    weight_decay=0.01,

    # Transformers 5.15.0:
    # 0~1 사이 float는 전체 step 대비 비율
    warmup_steps=0.1,

    lr_scheduler_type="linear",

    max_grad_norm=1.0,

    # -------------------------
    # Optimizer
    # -------------------------
    optim="adamw_torch",

    # -------------------------
    # Evaluation / Checkpoint
    # -------------------------
    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=2,

    # -------------------------
    # Logging
    # -------------------------
    logging_strategy="steps",
    logging_steps=50,
    logging_first_step=True,

    report_to="none",

    # -------------------------
    # Reproducibility
    # -------------------------
    seed=42,
    data_seed=42,

    # -------------------------
    # MPS
    # -------------------------
    fp16=False,
    bf16=False,

    dataloader_pin_memory=False,

    torch_empty_cache_steps=50,
)

### 29.11 Baseline 학습 설정 확인

정의한 Baseline 학습 설정이 의도한 값으로 적용되었는지 확인한다.

Apple Silicon 환경에서는 실제 Device가 `mps`로 설정되었는지도 함께 확인한다.

Baseline에서는 Device Batch Size를 2로 설정하고 Gradient Accumulation을 8회 적용하여 Effective Batch Size 16으로 학습한다.

In [21]:
effective_batch_size = (
    training_args.per_device_train_batch_size
    * training_args.gradient_accumulation_steps
)


print(
    "BASELINE TRAINING CONFIG"
)

print("=" * 60)

print(
    f"Epochs                  : "
    f"{training_args.num_train_epochs}"
)

print(
    f"Train batch size        : "
    f"{training_args.per_device_train_batch_size}"
)

print(
    f"Gradient accumulation   : "
    f"{training_args.gradient_accumulation_steps}"
)

print(
    f"Effective batch size    : "
    f"{effective_batch_size}"
)

print(
    f"Eval batch size         : "
    f"{training_args.per_device_eval_batch_size}"
)

print(
    f"Learning rate           : "
    f"{training_args.learning_rate}"
)

print(
    f"Weight decay            : "
    f"{training_args.weight_decay}"
)

print(
    f"Warmup                  : "
    f"{training_args.warmup_steps}"
)

print(
    f"Optimizer               : "
    f"{training_args.optim}"
)

print(
    f"Device                  : "
    f"{training_args.device}"
)

BASELINE TRAINING CONFIG
Epochs                  : 3
Train batch size        : 2
Gradient accumulation   : 8
Effective batch size    : 16
Eval batch size         : 2
Learning rate           : 3e-05
Weight decay            : 0.01
Warmup                  : 0.1
Optimizer               : OptimizerNames.ADAMW_TORCH
Device                  : mps


### 29.12 Trainer Smoke Test Dataset 구성

Forward Pass는 정상적으로 동작했지만 실제 Fine-tuning에는 Backward Pass와 Optimizer Update가 포함된다.

전체 Dataset을 학습하기 전에 작은 Dataset으로 10 Step의 Smoke Test를 수행한다.

Smoke Test에서는 학습 Pipeline 자체의 안정성을 확인하는 것이 목적이므로 비교적 짧은 Sequence를 우선 사용한다.

확인할 항목은 다음과 같다.

- MPS Backward Pass
- AdamW Optimizer
- Gradient Accumulation
- Dynamic Padding
- Trainer Training Loop
- Loss 계산

In [23]:
from torch.utils.data import Subset


SMOKE_TRAIN_SIZE = 128


shortest_train_indices = sorted(
    range(
        len(train_records)
    ),
    key=lambda index:
        len(
            train_records[
                index
            ]["input_ids"]
        ),
)[:SMOKE_TRAIN_SIZE]


smoke_train_dataset = Subset(
    train_dataset,
    shortest_train_indices,
)


smoke_lengths = [
    len(
        train_records[index][
            "input_ids"
        ]
    )
    for index
    in shortest_train_indices
]


print(
    f"Smoke samples       : "
    f"{len(smoke_train_dataset):,}"
)

print(
    f"Min sequence length : "
    f"{min(smoke_lengths)}"
)

print(
    f"Max sequence length : "
    f"{max(smoke_lengths)}"
)

Smoke samples       : 128
Min sequence length : 13
Max sequence length : 14


### 29.13 Smoke Test 모델 구성

Smoke Test에서 Weight Update가 발생하므로 실제 Baseline 학습에 사용할 모델과 별도의 모델을 생성한다.

Smoke Test가 끝난 뒤 해당 모델은 폐기하고 실제 Baseline 학습에서는 사전학습된 mmBERT를 다시 로드한다.

In [24]:
from transformers import (
    AutoModelForTokenClassification,
)


smoke_model = (
    AutoModelForTokenClassification
    .from_pretrained(
        PRIMARY_BACKBONE,

        num_labels=NUM_LABELS,

        id2label=id2label,
        label2id=label2id,
    )
)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForTokenClassification LOAD REPORT from: jhu-clsp/mmBERT-base
Key               | Status     | 
------------------+------------+-
decoder.weight    | UNEXPECTED | 
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [25]:
smoke_training_args = TrainingArguments(
    output_dir=(
        "../models/mmbert_smoke_test"
    ),

    max_steps=10,

    per_device_train_batch_size=2,

    gradient_accumulation_steps=2,

    learning_rate=3e-5,

    optim="adamw_torch",

    logging_strategy="steps",
    logging_steps=1,
    logging_first_step=True,

    eval_strategy="no",
    save_strategy="no",

    report_to="none",

    seed=42,

    fp16=False,
    bf16=False,

    dataloader_pin_memory=False,

    torch_empty_cache_steps=5,
)

In [26]:
from transformers import Trainer


smoke_trainer = Trainer(
    model=smoke_model,

    args=smoke_training_args,

    train_dataset=smoke_train_dataset,

    data_collator=data_collator,
)

### 29.16 Trainer Training Smoke Test

구성한 Smoke Test Dataset과 별도 mmBERT 모델을 이용하여 실제 10 Step Fine-tuning을 수행한다.

이 단계에서는 모델 성능 자체를 평가하지 않는다.

10 Step이 MPS 오류, Out Of Memory, NaN Loss 없이 완료되면 전체 Baseline 학습 Pipeline이 정상적으로 동작한다고 판단한다.

In [27]:
smoke_train_result = (
    smoke_trainer.train()
)

Step,Training Loss
1,8.108091
2,0.665885
3,0.004773
4,0.000057
5,0.000001
6,0.000001
7,0.000000
8,0.000000
9,0.000000
10,0.000000


In [28]:
print(
    "TRAINING SMOKE TEST RESULT"
)

print("=" * 60)

print(
    f"Training loss : "
    f"{smoke_train_result.training_loss:.6f}"
)

print(
    f"Global steps  : "
    f"{smoke_train_result.global_step}"
)

TRAINING SMOKE TEST RESULT
Training loss : 0.877881
Global steps  : 10


### 29.17 Smoke Test 정리

10 Step Training Smoke Test가 정상적으로 완료되었다.

이를 통해 다음 항목이 현재 Apple Silicon MPS 환경에서 정상적으로 동작함을 확인하였다.

- mmBERT Forward / Backward Pass
- AdamW Optimizer
- Gradient Accumulation
- Dynamic Padding
- Hugging Face Trainer
- Token Classification Loss 계산

Smoke Test 모델은 실제 Baseline 모델과 분리되어 있으므로 제거하고 MPS Memory를 정리한 뒤 새로운 사전학습 mmBERT 모델로 전체 학습을 시작한다.

In [29]:
import gc


del smoke_trainer
del smoke_model

gc.collect()


if torch.backends.mps.is_available():
    torch.mps.empty_cache()


print(
    "Smoke test resources cleared."
)

Smoke test resources cleared.


### 29.18 Baseline 모델 초기화

Smoke Test에서 Weight가 업데이트된 모델을 재사용하지 않는다.

동일한 사전학습 mmBERT Backbone에서 새로운 Token Classification 모델을 다시 로드하여 실제 Baseline Fine-tuning을 시작한다.

Token Classification Head는 25개의 BIO Label을 예측하도록 구성한다.

In [30]:
baseline_model = (
    AutoModelForTokenClassification
    .from_pretrained(
        PRIMARY_BACKBONE,

        num_labels=NUM_LABELS,

        id2label=id2label,
        label2id=label2id,
    )
)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForTokenClassification LOAD REPORT from: jhu-clsp/mmBERT-base
Key               | Status     | 
------------------+------------+-
decoder.weight    | UNEXPECTED | 
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### 29.19 Baseline Trainer 구성

전체 Train Dataset과 Validation Dataset을 사용하여 Baseline Trainer를 구성한다.

Validation은 Epoch마다 수행하며 `eval_loss`가 가장 낮은 Checkpoint를 Best Model로 선택한다.

Test Dataset은 학습 및 모델 선택에 사용하지 않는다.

In [31]:
baseline_trainer = Trainer(
    model=baseline_model,

    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=validation_dataset,

    data_collator=data_collator,
)

In [32]:
import math


train_batches_per_epoch = math.ceil(
    len(train_dataset)
    / training_args.per_device_train_batch_size
)


optimizer_steps_per_epoch = math.ceil(
    train_batches_per_epoch
    / training_args.gradient_accumulation_steps
)


total_optimizer_steps = (
    optimizer_steps_per_epoch
    * int(
        training_args.num_train_epochs
    )
)


print(
    "BASELINE TRAINING SIZE"
)

print("=" * 60)

print(
    f"Train samples             : "
    f"{len(train_dataset):,}"
)

print(
    f"Validation samples        : "
    f"{len(validation_dataset):,}"
)

print(
    f"Batches / epoch           : "
    f"{train_batches_per_epoch:,}"
)

print(
    f"Optimizer steps / epoch   : "
    f"{optimizer_steps_per_epoch:,}"
)

print(
    f"Total optimizer steps     : "
    f"{total_optimizer_steps:,}"
)

BASELINE TRAINING SIZE
Train samples             : 44,626
Validation samples        : 5,163
Batches / epoch           : 22,313
Optimizer steps / epoch   : 2,790
Total optimizer steps     : 8,370


### 29.21 mmBERT Baseline Fine-tuning

전체 Train Dataset을 이용하여 mmBERT Token Classification 모델을 3 Epoch Fine-tuning한다.

학습 중 Epoch마다 Validation Loss를 계산하고 가장 낮은 Validation Loss를 기록한 Checkpoint를 Best Model로 선택한다.

현재 단계에서는 Baseline 성능 확보가 목적이므로 별도의 Hyperparameter Tuning은 수행하지 않는다.

In [34]:
baseline_train_result = (
    baseline_trainer.train()
)

Epoch,Training Loss,Validation Loss
1,0.000001,0.000000
2,0.000001,0.000006
3,0.000000,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [35]:
print(
    "BASELINE TRAINING RESULT"
)

print("=" * 60)

print(
    f"Training loss   : "
    f"{baseline_train_result.training_loss:.10f}"
)

print(
    f"Global steps    : "
    f"{baseline_train_result.global_step:,}"
)

print(
    f"Best checkpoint : "
    f"{baseline_trainer.state.best_model_checkpoint}"
)

print(
    f"Best metric     : "
    f"{baseline_trainer.state.best_metric}"
)

BASELINE TRAINING RESULT
Training loss   : 0.0041260482
Global steps    : 8,370
Best checkpoint : ../models/mmbert_baseline/checkpoint-8370
Best metric     : 1.7217381298451073e-08


In [36]:
from collections import Counter


validation_label_counts = Counter()

for record in validation_records:
    for label_id in record["labels"]:
        validation_label_counts[label_id] += 1


print(
    "VALIDATION LABEL DISTRIBUTION"
)

print("=" * 60)


for label_id, count in sorted(
    validation_label_counts.items()
):
    label_name = (
        "IGNORE"
        if label_id == IGNORE_INDEX
        else id2label[label_id]
    )

    print(
        f"{label_name:<30} "
        f"{count:>12,}"
    )

VALIDATION LABEL DISTRIBUTION
IGNORE                               10,326
O                                   627,892
B-EMAIL                                  86
I-EMAIL                                 595
B-PHONE                                  25
I-PHONE                                 329
B-PERSON                                 90
I-PERSON                                149
B-IP_ADDRESS                          1,993
I-IP_ADDRESS                         26,370
B-TOKEN                                 562
I-TOKEN                              25,057
B-API_KEY                               174
I-API_KEY                             4,385
B-SESSION_ID                            561
I-SESSION_ID                         16,185
B-PASSWORD                              213
I-PASSWORD                            3,421
B-SECRET                                155
I-SECRET                              5,240
B-PRIVATE_KEY                            32
I-PRIVATE_KEY                         3,097
B-

In [37]:
import torch


baseline_model = (
    baseline_trainer.model
)

baseline_model.eval()


sample_index = 0

sample = validation_dataset[
    sample_index
]


batch = data_collator(
    [sample]
)

batch = {
    key: value.to(device)
    for key, value
    in batch.items()
}


with torch.no_grad():
    outputs = baseline_model(
        input_ids=batch[
            "input_ids"
        ],
        attention_mask=batch[
            "attention_mask"
        ],
    )


pred_ids = (
    outputs.logits
    .argmax(dim=-1)[0]
    .cpu()
    .tolist()
)


gold_ids = (
    batch["labels"][0]
    .cpu()
    .tolist()
)


tokens = tokenizer.convert_ids_to_tokens(
    batch["input_ids"][0]
    .cpu()
    .tolist()
)


for token, gold_id, pred_id in zip(
    tokens,
    gold_ids,
    pred_ids,
):
    if gold_id == IGNORE_INDEX:
        continue

    gold_label = id2label[
        gold_id
    ]

    pred_label = id2label[
        pred_id
    ]

    if (
        gold_label != "O"
        or pred_label != "O"
    ):
        print(
            f"{token:<25} "
            f"Gold={gold_label:<25} "
            f"Pred={pred_label}"
        )

▁                         Gold=B-IP_ADDRESS              Pred=B-IP_ADDRESS
1                         Gold=I-IP_ADDRESS              Pred=I-IP_ADDRESS
9                         Gold=I-IP_ADDRESS              Pred=I-IP_ADDRESS
2                         Gold=I-IP_ADDRESS              Pred=I-IP_ADDRESS
.                         Gold=I-IP_ADDRESS              Pred=I-IP_ADDRESS
0                         Gold=I-IP_ADDRESS              Pred=I-IP_ADDRESS
.                         Gold=I-IP_ADDRESS              Pred=I-IP_ADDRESS
2                         Gold=I-IP_ADDRESS              Pred=I-IP_ADDRESS
.                         Gold=I-IP_ADDRESS              Pred=I-IP_ADDRESS
2                         Gold=I-IP_ADDRESS              Pred=I-IP_ADDRESS
0                         Gold=I-IP_ADDRESS              Pred=I-IP_ADDRESS
